# EDA-5 — Signal Stability Analysis

Orchestration notebook. Executes governed block-distribution stability analysis across all governed signals and positions.

**Outputs:**
- `signals/eda/findings/eda_05_signal_stability.csv` — per-block distribution statistics
- `signals/eda/findings/eda_05_pooling_decisions.csv` — one pooling decision per (signal, position)

**Depends on:** `core.signals.stability`, `dal.prepared.analytical_dataset`

All classification logic is imported — no inline thresholds or vocabulary.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from dal.config import DB_PATH
from dal.curated.player_gameweek_spine import build_player_gameweek_spine
from dal.prepared.analytical_dataset import (
    build_prepared_dataset,
    GOVERNED_SIGNAL_COLUMNS,
)
from core.signals.stability import (
    BLOCK_HOMOGENEITY_VALUES,
    DEFAULT_GW_BLOCKS,
    POOLING_DECISION_VALUES,
    classify_block_homogeneity,
    compute_signal_block_distributions,
    flag_pooling_decision,
)

## Paths and constants

All output paths are defined here. `DATA_CUTOFF_GW` is the only notebook-local parameter: the upper GW bound for the prepared dataset.

In [ ]:
FINDINGS_DIR = Path("../findings")
FINDINGS_DIR.mkdir(parents=True, exist_ok=True)

STABILITY_OUTPUT_PATH = FINDINGS_DIR / "eda_05_signal_stability.csv"
POOLING_OUTPUT_PATH   = FINDINGS_DIR / "eda_05_pooling_decisions.csv"

# Upper GW bound for the prepared dataset. Set to the last completed gameweek.
DATA_CUTOFF_GW: int = 38

# Governed positions — matches POSITION_CODE_MAP values in analytical_dataset.
POSITIONS: list[str] = ["GK", "DEF", "MID", "FWD"]

# Governed signal set — sourced directly from the prepared dataset contract.
SIGNALS: list[str] = list(GOVERNED_SIGNAL_COLUMNS)

print(f"DB path:         {DB_PATH}")
print(f"Data cutoff GW:  {DATA_CUTOFF_GW}")
print(f"Positions:       {POSITIONS}")
print(f"Signal count:    {len(SIGNALS)}")
print(f"GW blocks:       {DEFAULT_GW_BLOCKS}")

## 1. Load prepared analytical dataset

Builds the governed prepared dataset from the curated spine. Fails early if required columns are missing.

In [ ]:
print(f"Loading spine from {DB_PATH} ...")
spine = build_player_gameweek_spine(DB_PATH)

print(f"Building prepared dataset (cutoff GW={DATA_CUTOFF_GW}) ...")
prepared = build_prepared_dataset(spine, data_cutoff_gw=DATA_CUTOFF_GW)

print(f"Prepared dataset: {len(prepared):,} rows × {len(prepared.columns)} columns")
print(f"GW range: {prepared['gw'].min()} – {prepared['gw'].max()}")
print(f"Position distribution:\n{prepared['position'].value_counts().to_string()}")

## 2. Validate required columns

Asserts that all governed signals and required structural columns exist in the prepared dataset before any analysis executes.

In [ ]:
REQUIRED_COLUMNS = {"gw", "position"} | set(SIGNALS)
missing_columns = REQUIRED_COLUMNS - set(prepared.columns)
if missing_columns:
    raise ValueError(
        f"Prepared dataset is missing required columns: {sorted(missing_columns)}\n"
        "Cannot proceed with stability analysis."
    )

# Filter signals to those actually present (some may be absent for valid reasons).
signals_present = [s for s in SIGNALS if s in prepared.columns]
signals_absent  = [s for s in SIGNALS if s not in prepared.columns]

if signals_absent:
    print(f"WARNING: {len(signals_absent)} governed signals not in prepared dataset: {signals_absent}")

# Cast pandas ExtensionDtype signal columns (Int64, Float64, boolean) to float64.
# numpy.percentile inside compute_signal_block_distributions requires standard numpy dtypes.
# This is data preparation, not analytical logic — values are preserved; NaN propagates correctly.
prepared = prepared.copy()
for col in signals_present:
    if hasattr(prepared[col].dtype, "numpy_dtype"):
        prepared[col] = prepared[col].astype(float)

print(f"Column validation passed. Signals to analyze: {len(signals_present)}")

## 3. Execute governed stability analysis

Runs `compute_signal_block_distributions` then `classify_block_homogeneity` and `flag_pooling_decision` for each (signal, position) pair. Uses `DEFAULT_GW_BLOCKS` from `analytics.signals.stability`.

In [ ]:
print("Computing block distributions ...")
block_stats = compute_signal_block_distributions(
    df=prepared,
    signals=signals_present,
    positions=POSITIONS,
    gw_column="gw",
    gw_blocks=DEFAULT_GW_BLOCKS,
)

print(f"Block distribution rows: {len(block_stats):,}  (signals × positions × blocks)")

pooling_rows: list[dict] = []

print("\nClassifying homogeneity and flagging pooling decisions ...")
for signal in signals_present:
    for position in POSITIONS:
        pair_stats = block_stats[
            (block_stats["signal"] == signal) & (block_stats["position"] == position)
        ]
        if pair_stats.empty:
            continue

        homogeneity = classify_block_homogeneity(pair_stats)
        pooling_decision = flag_pooling_decision(homogeneity)

        pooling_rows.append({
            "signal":          signal,
            "position":        position,
            "homogeneity":     homogeneity,
            "pooling_decision": pooling_decision,
        })
        print(f"  {signal} × {position}: {homogeneity} → {pooling_decision}")

pooling_df = pd.DataFrame(pooling_rows)
print(f"\nPooling decisions: {len(pooling_df)} rows")

## 4. Validate outputs before writing

Asserts governed vocabulary membership, no null signal names, and no duplicate (signal, position) rows in the pooling decisions output.

In [ ]:
# Assert no null signal names in block_stats.
null_signals_block = block_stats["signal"].isna().sum()
assert null_signals_block == 0, (
    f"block_stats has {null_signals_block} null signal names"
)

# Assert homogeneity values belong to governed vocabulary.
invalid_homogeneity = set(pooling_df["homogeneity"].unique()) - BLOCK_HOMOGENEITY_VALUES
assert not invalid_homogeneity, (
    f"pooling_df contains unrecognized homogeneity values: {invalid_homogeneity}\n"
    f"Governed vocabulary: {sorted(BLOCK_HOMOGENEITY_VALUES)}"
)

# Assert pooling_decision values belong to governed vocabulary.
invalid_pooling = set(pooling_df["pooling_decision"].unique()) - POOLING_DECISION_VALUES
assert not invalid_pooling, (
    f"pooling_df contains unrecognized pooling_decision values: {invalid_pooling}\n"
    f"Governed vocabulary: {sorted(POOLING_DECISION_VALUES)}"
)

# Assert no duplicate (signal, position) rows in pooling decisions.
duplicates = int(pooling_df[["signal", "position"]].duplicated().sum())
assert duplicates == 0, (
    f"pooling_df has {duplicates} duplicate (signal, position) rows"
)

# Assert no null signal names in pooling_df.
null_signals_pooling = pooling_df["signal"].isna().sum()
assert null_signals_pooling == 0, (
    f"pooling_df has {null_signals_pooling} null signal names"
)

print("All output validations passed.")

## 5. Write governed outputs

In [ ]:
block_stats.to_csv(STABILITY_OUTPUT_PATH, index=False)
print(f"Written: {STABILITY_OUTPUT_PATH}  ({len(block_stats)} rows)")

pooling_df.to_csv(POOLING_OUTPUT_PATH, index=False)
print(f"Written: {POOLING_OUTPUT_PATH}  ({len(pooling_df)} rows)")

## 6. Interpretation summary

Summary counts only. No speculative interpretation.

In [ ]:
print("=" * 60)
print("EDA-5 — Signal Stability Summary")
print("=" * 60)

print("\nHomogeneity classification counts (signal × position pairs):")
homogeneity_counts = pooling_df["homogeneity"].value_counts().reindex(
    sorted(BLOCK_HOMOGENEITY_VALUES), fill_value=0
)
for label, count in homogeneity_counts.items():
    print(f"  {label:<20} {count}")

print("\nPooling decision counts (signal × position pairs):")
pooling_counts = pooling_df["pooling_decision"].value_counts().reindex(
    sorted(POOLING_DECISION_VALUES), fill_value=0
)
for label, count in pooling_counts.items():
    print(f"  {label:<30} {count}")

print("\nOutputs:")
print(f"  {STABILITY_OUTPUT_PATH}")
print(f"  {POOLING_OUTPUT_PATH}")
print("=" * 60)